# MNIST MLP3 — MuonClip polar Jacobian: analytic vs numerical WeightWatcher fits

For each matrix checkpoint \(W=U\Sigma V^\top\), define the polar projection \(\Pi(W)=UV^\top\), its Fréchet derivative \(J_W=D\Pi(W)\), and the Gram operator \(G_W=J_W^\ast J_W\). This notebook compares two spectra from the **same single checkpoint** and sends both raw positive spectra to WeightWatcher's own `WW_powerlaw.pl_fit`.

**Analytic derivative.** With \(A=U^\top E V\),
\[
D\Pi_W[E]=U\Omega V^\top + D\Pi_W[E]_\perp,
\qquad
\Omega_{ij}=\frac{A_{ij}-A_{ji}}{\sigma_i+\sigma_j},\;\Omega_{ii}=0.
\]
For \(m>n\), \(D\Pi_W[E]_\perp=(I-UU^\top)EV\Sigma^{-1}V^\top\); for \(n>m\), \(D\Pi_W[E]_\perp=U\Sigma^{-1}U^\top E(I-VV^\top)\). Therefore the nonzero eigenvalues of \(G_W\) are
\[
\boxed{\lambda_{ij}^{\rm rot}=4/(\sigma_i+\sigma_j)^2},\quad i<j,
\qquad
\boxed{\lambda_i^\perp=1/\sigma_i^2}
\]
with \(\lambda_i^\perp\) repeated \(|m-n|\) times.

**Numerical single-checkpoint method.** Draw isotropic directions \(E_a=Z_a/\|Z_a\|_F\), compute
\[
X_a=\frac{\Pi(W+\varepsilon E_a)-\Pi(W-\varepsilon E_a)}{2\varepsilon},
\]
stack \(\operatorname{vec}(X_a)\) into columns of \(X\), and form \(G_{\rm probe}=X^\top X\). Its positive eigenvalues are a finite-probe local-response spectrum. We compare WeightWatcher \(\alpha\), \(D\), \(x_{\min}\), and tail size against the analytic fit. No ad-hoc spectrum normalization or hand-written PL fitting is used.


In [ ]:
from pathlib import Path
import os,sys,math,random
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,transforms
from IPython.display import display
import matplotlib.pyplot as plt

ROOT=None
for q in [Path.cwd(),*Path.cwd().parents]:
    if (q/'baseline'/'rg_baselines').is_dir(): ROOT=(q/'baseline').resolve(); break
    if (q/'rg_baselines').is_dir(): ROOT=q.resolve(); break
if ROOT is None: raise RuntimeError('Run from CalculatedContent/rg_optimizers')
sys.path.insert(0,str(ROOT))
from rg_baselines import MLP3,DEFAULT_BASELINE_SEEDS,MNIST_REFERENCE_SUITE_SLUG
from rg_baselines.polar_jacobian import analytic_gram_spectrum,numerical_probe_gram_spectrum,weightwatcher_pl_fit,finite_difference_error,probe_convergence

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR=Path(os.environ.get('RG_BASELINE_DATA_DIR',ROOT/'data')).expanduser().resolve()
RUN_ROOT=Path(os.environ.get('RG_BASELINE_RUN_ROOT',ROOT/'runs')).expanduser().resolve()/MNIST_REFERENCE_SUITE_SLUG/'muonclip_polar_jacobian_analytic_vs_numeric'; RUN_ROOT.mkdir(parents=True,exist_ok=True)
EPOCHS=int(os.environ.get('MUONCLIP_POLAR_EPOCHS','30')); BATCH=int(os.environ.get('MUONCLIP_POLAR_BATCH','256'))
SEEDS=tuple(int(x) for x in os.environ.get('MUONCLIP_POLAR_SEEDS',','.join(map(str,DEFAULT_BASELINE_SEEDS))).split(','))
PROBES=int(os.environ.get('POLAR_NUMERIC_PROBES','128')); EPS=float(os.environ.get('POLAR_NUMERIC_EPS_REL','1e-5')); NUMERIC_EVERY=int(os.environ.get('POLAR_NUMERIC_EVERY','0'))
print(DEVICE,RUN_ROOT)


In [ ]:
MATRIX_LR=2e-3; AUX_LR=2e-3; MOMENTUM=.95; WEIGHT_DECAY=1e-2; RMS_SCALE=.20; GRAD_CLIP=1.0
@torch.no_grad()
def zeropower(g,steps=5,eps=1e-7):
    t=g.shape[0]>g.shape[1]; x=g.T if t else g; x=x.float()/torch.linalg.vector_norm(x.float()).clamp_min(eps); a,b,c=3.4445,-4.7750,2.0315
    for _ in range(steps): q=x@x.T; x=a*x+(b*q+c*(q@q))@x
    return x.T if t else x
class MuonClip(torch.optim.Optimizer):
    def __init__(self,params): super().__init__(params,dict(lr=MATRIX_LR,momentum=MOMENTUM,weight_decay=WEIGHT_DECAY))
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                b=self.state[p].setdefault('momentum_buffer',torch.zeros_like(p.grad)); b.mul_(group['momentum']).add_(p.grad)
                u=zeropower(b).to(p.dtype); u.mul_(RMS_SCALE*math.sqrt(max(p.shape))); p.mul_(1-group['lr']*group['weight_decay']); p.add_(u,alpha=-group['lr'])


In [ ]:
# Verify the analytic Fréchet formula numerically on small rectangular matrices.
rng=np.random.default_rng(123)
for shape in [(3,3),(4,3),(3,5)]:
    W=rng.normal(size=shape); E=rng.normal(size=shape); err=finite_difference_error(W,E)
    print(shape,err); assert err<1e-7


In [ ]:
transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,),(0.3081,))])
full=datasets.MNIST(str(DATA_DIR),train=True,download=True,transform=transform); test=datasets.MNIST(str(DATA_DIR),train=False,download=True,transform=transform)
g=torch.Generator().manual_seed(20_260_807); perm=torch.randperm(len(full),generator=g).tolist(); val_idx,train_idx=perm[:5000],perm[5000:]; train,val=Subset(full,train_idx),Subset(full,val_idx)
def loaders(seed):
    tg=torch.Generator().manual_seed(seed+101); kw=dict(batch_size=BATCH,num_workers=0,pin_memory=DEVICE.type=='cuda')
    return DataLoader(train,shuffle=True,generator=tg,**kw),DataLoader(train,shuffle=False,**kw),DataLoader(val,shuffle=False,**kw),DataLoader(test,shuffle=False,**kw)
@torch.inference_mode()
def evaluate(model,loader):
    model.eval(); loss=correct=n=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); z=model(x); loss+=float(F.cross_entropy(z,y,reduction='sum').cpu()); correct+=int((z.argmax(1)==y).sum().cpu()); n+=y.numel()
    return loss/n,correct/n


In [ ]:
def seed_all(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
def numerical_due(epoch): return epoch in {0,EPOCHS} or (NUMERIC_EVERY>0 and epoch%NUMERIC_EVERY==0)
def analyze(model,seed,epoch):
    rows=[]; spectra={}
    for j,name in enumerate(('fc1','fc2','fc3')):
        W=getattr(model,name).weight.detach().cpu().numpy(); a,z=analytic_gram_spectrum(W); af=weightwatcher_pl_fit(a)
        rows.append(dict(seed=seed,epoch=epoch,layer=name,method='analytic',zero_modes=z,positive_modes=len(a),probes=np.nan,epsilon=np.nan,**af)); spectra[f'analytic__seed_{seed}__epoch_{epoch:03d}__{name}']=a
        if numerical_due(epoch):
            n,meta=numerical_probe_gram_spectrum(W,probes=PROBES,eps_rel=EPS,seed=918273+100000*seed+100*epoch+j); nf=weightwatcher_pl_fit(n)
            rows.append(dict(seed=seed,epoch=epoch,layer=name,method='numerical_finite_difference',zero_modes=np.nan,positive_modes=len(n),probes=PROBES,epsilon=meta['epsilon'],**nf)); spectra[f'numeric__seed_{seed}__epoch_{epoch:03d}__{name}']=n
    return rows,spectra

perf=[]; fits=[]; spectra={}
for seed in SEEDS:
    seed_all(seed); tr,tre,va,te=loaders(seed); model=MLP3().to(DEVICE); matrices=[p for p in model.parameters() if p.ndim==2]; biases=[p for p in model.parameters() if p.ndim!=2]; muon=MuonClip(matrices); aux=torch.optim.AdamW(biases,lr=AUX_LR,weight_decay=WEIGHT_DECAY)
    r,s=analyze(model,seed,0); fits+=r; spectra.update(s)
    for epoch in range(1,EPOCHS+1):
        model.train()
        for x,y in tr:
            x,y=x.to(DEVICE),y.to(DEVICE); muon.zero_grad(set_to_none=True); aux.zero_grad(set_to_none=True); loss=F.cross_entropy(model(x),y); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); muon.step(); aux.step()
        tl,ta=evaluate(model,tre); vl,vaa=evaluate(model,va); ql,qa=evaluate(model,te); perf.append(dict(seed=seed,epoch=epoch,train_loss=tl,validation_loss=vl,test_loss=ql,train_accuracy=ta,validation_accuracy=vaa,test_accuracy=qa)); r,s=analyze(model,seed,epoch); fits+=r; spectra.update(s); print(seed,epoch,vl)
performance=pd.DataFrame(perf); gram_fits=pd.DataFrame(fits); performance.to_csv(RUN_ROOT/'performance_by_epoch_and_seed.csv',index=False); gram_fits.to_csv(RUN_ROOT/'polar_jacobian_gram_weightwatcher_analytic_vs_numeric.csv',index=False); np.savez_compressed(RUN_ROOT/'polar_jacobian_gram_spectra_analytic_vs_numeric.npz',**spectra)


## WeightWatcher fit comparison


In [ ]:
a=gram_fits[gram_fits.method.eq('analytic')]; n=gram_fits[gram_fits.method.eq('numerical_finite_difference')]
cmp=n.merge(a,on=['seed','epoch','layer'],suffixes=('_numeric','_analytic'),validate='one_to_one'); cmp['alpha_difference']=cmp.alpha_numeric-cmp.alpha_analytic; cmp['alpha_abs_difference']=cmp.alpha_difference.abs(); cmp.to_csv(RUN_ROOT/'polar_jacobian_weightwatcher_fit_comparison.csv',index=False)
display(cmp[['seed','epoch','layer','probes_numeric','alpha_analytic','alpha_numeric','alpha_difference','D_analytic','D_numeric','xmin_analytic','xmin_numeric','tail_evals_analytic','tail_evals_numeric']])
if not cmp.empty:
    fig,ax=plt.subplots(figsize=(7,6))
    for layer,f in cmp.groupby('layer'): ax.scatter(f.alpha_analytic,f.alpha_numeric,label=layer)
    vals=np.r_[cmp.alpha_analytic,cmp.alpha_numeric]; lo,hi=np.nanmin(vals),np.nanmax(vals); ax.plot([lo,hi],[lo,hi],'--'); ax.set(xlabel='analytic WW alpha',ylabel='numerical WW alpha',title='Polar Jacobian Gram PL fit'); ax.grid(alpha=.25); ax.legend(frameon=False); plt.show()


## Single-checkpoint probe convergence

For any selected checkpoint matrix `W`, use the helper below to compare the analytic WeightWatcher fit against numerical finite-difference fits with increasing probe count.


In [ ]:
# Example:
# W=model.fc2.weight.detach().cpu().numpy()
# display(pd.DataFrame(probe_convergence(W,counts=(16,32,64,128,256),eps_rel=EPS)))


## Outputs

`polar_jacobian_gram_weightwatcher_analytic_vs_numeric.csv` contains both methods' WeightWatcher fits. `polar_jacobian_gram_spectra_analytic_vs_numeric.npz` contains the raw positive eigenvalues actually passed to WeightWatcher. `polar_jacobian_weightwatcher_fit_comparison.csv` places analytic and numerical `alpha`, `D`, `xmin`, and tail counts side by side.
